In [1]:
import requests
r = requests.get("http://www.khs.go.kr/cha/SearchKindOpenapiList.do", params={
    "pageUnit": 1,
    "pageIndex": 1,
    "ccbaCncl": "N",
    "ccbaCtcd": "11"
})
print(r.text)

<?xml version="1.0" encoding="utf-8"?>







<result>
<totalCnt>2115</totalCnt>
<pageUnit>1</pageUnit>
<pageIndex>1</pageIndex>	

<item>
<sn>1</sn>
<no>1</no>
<ccmaName><![CDATA[국보]]></ccmaName>
<ccbaMnm1><![CDATA[서울 숭례문]]></ccbaMnm1>
<ccbaMnm2><![CDATA[서울 崇禮門]]></ccbaMnm2>
<ccbaCtcdNm><![CDATA[서울]]></ccbaCtcdNm>
<ccsiName><![CDATA[중구]]></ccsiName>
<ccbaAdmin><![CDATA[국가유산청 덕수궁관리소]]></ccbaAdmin>
<ccbaKdcd>11</ccbaKdcd>
<ccbaCtcd>11</ccbaCtcd>
<ccbaAsno>0000010000000</ccbaAsno>
<ccbaCncl>N</ccbaCncl>
<ccbaCpno>1111100010000</ccbaCpno>
<longitude>126.975312652739</longitude>
<latitude>37.559975221378</latitude>
<regDt>2025-06-26 18:46:08</regDt>

</item>
	
</result>



# # 바로 실행
df = main()

# 또는 개별 실행
heritage_df = collect_national_heritage_all()  # 국가유산만
gung_df = collect_gung_all()  # 궁궐·종묘만

In [5]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import json
import logging
import xml.etree.ElementTree as ET
from urllib.parse import urlencode

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# === 시도 코드 목록 (전국 대상) === #
SIDO_CODES = [
    "11", "21", "22", "23", "24", "25", "26", "45",
    "31", "32", "33", "34", "35", "36", "37", "38", "50"
]

# 시도 코드명 매핑
SIDO_NAMES = {
    "11": "서울", "21": "부산", "22": "대구", "23": "인천", 
    "24": "광주", "25": "대전", "26": "울산", "45": "세종",
    "31": "경기", "32": "강원", "33": "충북", "34": "충남", 
    "35": "전북", "36": "전남", "37": "경북", "38": "경남", "50": "제주"
}

def safe_api_call(func, *args, **kwargs):
    """API 호출을 안전하게 처리하는 래퍼 함수"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except requests.exceptions.RequestException as e:
            logger.warning(f"API 호출 실패 (시도 {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # 지수 백오프
            else:
                logger.error(f"최대 재시도 횟수 초과: {e}")
                return None
        except Exception as e:
            logger.error(f"예상치 못한 오류: {e}")
            return None

def parse_xml_response(xml_text):
    """XML 응답을 파싱하여 딕셔너리로 변환"""
    try:
        root = ET.fromstring(xml_text)
        
        # 공통적인 XML 구조 처리
        result = {"data": []}
        
        # item 태그들을 찾아서 처리
        items = root.findall('.//item')
        if not items:
            # 다른 구조일 수 있으므로 모든 하위 요소 확인
            items = root.findall('.//*')
            if len(items) <= 3:  # 루트 요소만 있는 경우
                return {"data": []}
        
        data_list = []
        for item in items:
            if item.tag == 'item' or len(list(item)) > 0:  # item 태그이거나 하위 요소가 있는 경우
                item_dict = {}
                for child in item:
                    item_dict[child.tag] = child.text if child.text else ""
                if item_dict:  # 빈 딕셔너리가 아닌 경우만 추가
                    data_list.append(item_dict)
        
        result["data"] = data_list
        return result
        
    except ET.ParseError as e:
        logger.error(f"XML 파싱 오류: {e}")
        return {"data": []}
    except Exception as e:
        logger.error(f"XML 처리 중 오류: {e}")
        return {"data": []}

def test_api_response(url, params):
    """API 응답 형식을 테스트하는 함수"""
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        logger.info(f"API 응답 테스트:")
        logger.info(f"Status Code: {response.status_code}")
        logger.info(f"Content-Type: {content_type}")
        logger.info(f"Response length: {len(response.text)}")
        logger.info(f"First 200 chars: {response.text[:200]}")
        
        return response.text, content_type
        
    except Exception as e:
        logger.error(f"API 테스트 실패: {e}")
        return None, None

# === 국가유산 목록 API === #
def fetch_heritage_list(page=1, per_page=50, ccbaCtcd='11'):
    """국가유산 목록을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchKindOpenapiList.do"
    params = {
        "pageUnit": per_page,
        "pageIndex": page,
        "ccbaCncl": "N",  # 해제되지 않은 유산만
        "ccbaCtcd": ccbaCtcd
    }
    
    logger.info(f"국가유산 목록 요청: 시도코드={ccbaCtcd}, 페이지={page}")
    
    # 첫 페이지인 경우 API 응답 형식 테스트
    if page == 1:
        test_response, test_content_type = test_api_response(url, params)
        if test_response is None:
            return {"data": []}
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        
        # 응답 내용 확인
        if not response.text.strip():
            logger.warning("빈 응답을 받았습니다.")
            return {"data": []}
        
        # Content-Type 확인 및 파싱
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            logger.info("XML 응답을 파싱합니다.")
            return parse_xml_response(response.text)
        elif 'application/json' in content_type:
            try:
                data = response.json()
                if isinstance(data, dict) and "result" in data:
                    return {"data": data.get("result", [])}
                elif isinstance(data, dict) and "data" in data:
                    return data
                elif isinstance(data, list):
                    return {"data": data}
                else:
                    return {"data": []}
            except json.JSONDecodeError as e:
                logger.error(f"JSON 파싱 오류: {e}")
                return {"data": []}
        else:
            # Content-Type이 명확하지 않은 경우, 내용을 보고 판단
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                logger.info("XML로 추정되는 응답을 파싱합니다.")
                return parse_xml_response(text)
            elif text.startswith('{') or text.startswith('['):
                try:
                    data = json.loads(text)
                    return {"data": data if isinstance(data, list) else [data]}
                except:
                    logger.warning("JSON 파싱 실패, 빈 결과 반환")
                    return {"data": []}
            else:
                logger.warning(f"알 수 없는 응답 형식: {content_type}")
                logger.debug(f"응답 내용: {text[:200]}")
                return {"data": []}
            
    except requests.exceptions.Timeout:
        logger.error("요청 시간 초과")
        return {"data": []}
    except requests.exceptions.RequestException as e:
        logger.error(f"요청 오류: {e}")
        return {"data": []}

# === 국가유산 상세 API === #
def fetch_heritage_detail(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 상세 정보를 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchKindOpenapiDt.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        if not response.text.strip():
            return {"data": {}}
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            # 상세 정보는 보통 단일 아이템이므로 첫 번째 아이템 반환
            data_list = parsed.get("data", [])
            return {"data": data_list[0] if data_list else {}}
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "result" in data:
                return {"data": data.get("result", {})}
            elif isinstance(data, dict) and "data" in data:
                return data
            else:
                return {"data": data if isinstance(data, dict) else {}}
        else:
            # 자동 감지
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                data_list = parsed.get("data", [])
                return {"data": data_list[0] if data_list else {}}
            else:
                return {"data": {}}
            
    except Exception as e:
        logger.warning(f"상세 정보 가져오기 실패: {e}")
        return {"data": {}}

# === 국가유산 이미지 API === #
def fetch_heritage_image(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 이미지 URL을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchImageOpenapi.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            result = parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "data" in data:
                result = data.get("data", [])
            elif isinstance(data, list):
                result = data
            else:
                result = []
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                result = parsed.get("data", [])
            else:
                result = []
        
        return result[0].get("imageUrl", "") if result else ""
        
    except Exception:
        return ""

# === 국가유산 동영상 API === #
def fetch_heritage_video(ccbaKdcd, ccbaAsno, ccbaCtcd):
    """국가유산 동영상 URL을 가져오는 함수"""
    url = "http://www.khs.go.kr/cha/SearchVideoOpenapi.do"
    params = {
        "ccbaKdcd": ccbaKdcd,
        "ccbaAsno": ccbaAsno,
        "ccbaCtcd": ccbaCtcd
    }
    
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            result = parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            if isinstance(data, dict) and "data" in data:
                result = data.get("data", [])
            elif isinstance(data, list):
                result = data
            else:
                result = []
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                result = parsed.get("data", [])
            else:
                result = []
            
        return result[0].get("videoUrl", "") if result else ""
        
    except Exception:
        return ""

# === 전국 국가유산 전체 수집 === #
def collect_national_heritage_all():
    """전국의 모든 국가유산 정보를 수집"""
    result = []
    total_collected = 0
    
    for ctcd in tqdm(SIDO_CODES, desc="전국 시도코드 순회"):
        sido_name = SIDO_NAMES.get(ctcd, ctcd)
        logger.info(f"{sido_name}({ctcd}) 지역 수집 시작")
        
        page = 1
        sido_count = 0
        
        while True:
            # API 호출
            data = safe_api_call(fetch_heritage_list, page=page, ccbaCtcd=ctcd)
            
            if data is None:
                logger.warning(f"{sido_name} {page}페이지 건너뜀")
                break
                
            items = data.get("data", [])
            if not items:
                logger.info(f"{sido_name} {page}페이지에서 더 이상 데이터 없음")
                break
            
            logger.info(f"{sido_name} {page}페이지: {len(items)}건 처리 중")
            
            for item in items:
                try:
                    # 상세 정보 가져오기
                    detail_data = safe_api_call(
                        fetch_heritage_detail, 
                        item.get("ccbaKdcd", ""), 
                        item.get("ccbaAsno", ""), 
                        item.get("ccbaCtcd", "")
                    )
                    
                    detail = detail_data.get("data", {}) if detail_data else {}
                    
                    # 이미지 및 동영상 URL 가져오기 (선택사항)
                    image_url = safe_api_call(
                        fetch_heritage_image,
                        item.get("ccbaKdcd", ""),
                        item.get("ccbaAsno", ""),
                        item.get("ccbaCtcd", "")
                    ) or ""
                    
                    video_url = safe_api_call(
                        fetch_heritage_video,
                        item.get("ccbaKdcd", ""),
                        item.get("ccbaAsno", ""),
                        item.get("ccbaCtcd", "")
                    ) or ""
                    
                    # 데이터 정리
                    heritage_item = {
                        "title": detail.get("ccbaMnm1") or item.get("ccbaMnm1", ""),
                        "title_hanja": detail.get("ccbaMnm2") or item.get("ccbaMnm2", ""),
                        "type": detail.get("ccmaName") or item.get("ccmaName", ""),
                        "category": detail.get("gcodeName", ""),
                        "location": detail.get("ccbaLcad") or item.get("ccbaLcto", ""),
                        "sido": sido_name,
                        "description": detail.get("content", ""),
                        "designation_date": detail.get("ccbaAsdt", ""),
                        "owner": detail.get("ccbaPoss", ""),
                        "manager": detail.get("ccbaAdmin", ""),
                        "era": detail.get("ccceName", ""),
                        "quantity": detail.get("ccbaQuan", ""),
                        "longitude": item.get("longitude", "0"),
                        "latitude": item.get("latitude", "0"),
                        "image_url": image_url,
                        "video_url": video_url,
                        "heritage_code": item.get("ccbaKdcd", ""),
                        "management_number": item.get("ccbaAsno", ""),
                        "sido_code": item.get("ccbaCtcd", "")
                    }
                    
                    result.append(heritage_item)
                    sido_count += 1
                    total_collected += 1
                    
                    # 진행 상황 출력 (100건마다)
                    if total_collected % 100 == 0:
                        logger.info(f"총 {total_collected}건 수집 완료")
                    
                except Exception as e:
                    logger.warning(f"항목 처리 중 오류: {e}")
                    continue
            
            page += 1
            time.sleep(0.5)  # API 부하 방지
        
        logger.info(f"{sido_name} 완료: {sido_count}건 수집")
    
    logger.info(f"국가유산 수집 완료: 총 {total_collected}건")
    return pd.DataFrame(result)

# === 궁궐·종묘 수집 === #
def fetch_gung_list(gung_number=1):
    """궁궐·종묘 목록을 가져오는 함수"""
    url = "https://www.heritage.go.kr/heri/gungDetail/gogungListOpenApi.do"
    params = {"gung_number": gung_number}
    
    logger.info(f"궁궐 목록 요청: 궁번호={gung_number}")
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        logger.info(f"궁궐 API 응답: Status={response.status_code}, Content-Type={response.headers.get('content-type')}")
        logger.info(f"응답 내용 미리보기: {response.text[:200]}")
        
        if not response.text.strip():
            logger.warning("빈 응답")
            return []
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            return parsed.get("data", [])
        elif 'application/json' in content_type:
            data = response.json()
            return data.get("data", []) if isinstance(data, dict) else data
        else:
            # 자동 감지
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                return parsed.get("data", [])
            elif text.startswith('{') or text.startswith('['):
                try:
                    data = json.loads(text)
                    return data.get("data", []) if isinstance(data, dict) else data
                except:
                    return []
            else:
                logger.warning(f"알 수 없는 궁궐 API 응답 형식")
                return []
                
    except Exception as e:
        logger.warning(f"궁궐 목록 가져오기 실패 (궁번호: {gung_number}): {e}")
        return []

def fetch_gung_detail(serial_number, gung_number, detail_code):
    """궁궐·종묘 상세 정보를 가져오는 함수"""
    url = "https://www.heritage.go.kr/heri/gungDetail/gogungDetailOpenApi.do"
    params = {
        "serial_number": serial_number,
        "gung_number": gung_number,
        "detail_code": detail_code
    }
    
    try:
        response = requests.get(url, params=params, timeout=20)
        response.raise_for_status()
        
        content_type = response.headers.get('content-type', '')
        
        if 'application/xml' in content_type or 'text/xml' in content_type:
            parsed = parse_xml_response(response.text)
            data_list = parsed.get("data", [])
            return data_list[0] if data_list else {}
        elif 'application/json' in content_type:
            data = response.json()
            return data.get("data", {}) if isinstance(data, dict) else data
        else:
            text = response.text.strip()
            if text.startswith('<?xml') or text.startswith('<'):
                parsed = parse_xml_response(text)
                data_list = parsed.get("data", [])
                return data_list[0] if data_list else {}
            else:
                return {}
                
    except Exception as e:
        logger.warning(f"궁궐 상세 정보 가져오기 실패: {e}")
        return {}

def collect_gung_all():
    """모든 궁궐·종묘 정보를 수집"""
    result = []
    gung_names = {1: "경복궁", 2: "창덕궁", 3: "창경궁", 4: "덕수궁", 5: "종묘"}
    
    for gung_number in tqdm(range(1, 6), desc="궁궐/종묘 수집"):
        gung_name = gung_names[gung_number]
        logger.info(f"{gung_name} 수집 시작")
        
        list_data = safe_api_call(fetch_gung_list, gung_number)
        
        if not list_data:
            logger.warning(f"{gung_name} 목록을 가져올 수 없습니다.")
            continue
        
        for item in list_data:
            try:
                detail = safe_api_call(
                    fetch_gung_detail,
                    item.get("serial_number"),
                    item.get("gung_number"),
                    item.get("detail_code")
                ) or {}
                
                gung_item = {
                    "title": detail.get("contents_kor") or item.get("contents_kor", ""),
                    "title_hanja": "",
                    "type": "궁궐·종묘",
                    "category": gung_name,
                    "location": "서울특별시",
                    "sido": "서울",
                    "description": detail.get("explanation_kor") or item.get("explanation_kor", ""),
                    "designation_date": "",
                    "owner": "",
                    "manager": "",
                    "era": "",
                    "quantity": "",
                    "longitude": "0",
                    "latitude": "0",
                    "image_url": detail.get("imgUrl") or item.get("imgUrl", ""),
                    "video_url": detail.get("moving", ""),
                    "heritage_code": "",
                    "management_number": "",
                    "sido_code": "11"
                }
                
                result.append(gung_item)
                
            except Exception as e:
                logger.warning(f"{gung_name} 항목 처리 중 오류: {e}")
                continue
        
        logger.info(f"{gung_name} 완료")
        time.sleep(1)  # API 부하 방지
    
    logger.info(f"궁궐·종묘 수집 완료: 총 {len(result)}건")
    return pd.DataFrame(result)

# === 메인 실행 함수 === #
def main():
    """메인 실행 함수"""
    logger.info("국가유산 데이터 수집 시작")
    
    # 국가유산 수집
    try:
        heritage_df = collect_national_heritage_all()
        logger.info(f"국가유산 DataFrame 생성 완료: {len(heritage_df)}건")
    except Exception as e:
        logger.error(f"국가유산 수집 중 오류: {e}")
        heritage_df = pd.DataFrame()
    
    # 궁궐·종묘 수집
    try:
        gung_df = collect_gung_all()
        logger.info(f"궁궐·종묘 DataFrame 생성 완료: {len(gung_df)}건")
    except Exception as e:
        logger.error(f"궁궐·종묘 수집 중 오류: {e}")
        gung_df = pd.DataFrame()
    
    # 데이터 통합
    if not heritage_df.empty and not gung_df.empty:
        final_df = pd.concat([heritage_df, gung_df], ignore_index=True)
    elif not heritage_df.empty:
        final_df = heritage_df
    elif not gung_df.empty:
        final_df = gung_df
    else:
        logger.error("수집된 데이터가 없습니다.")
        return pd.DataFrame()
    
    # 데이터 정리
    final_df = final_df.fillna("")  # NaN 값을 빈 문자열로 변경
    final_df = final_df.drop_duplicates(subset=['title', 'location'], keep='first')  # 중복 제거
    
    # CSV 저장
    filename = "heritage_rag_full.csv"
    final_df.to_csv(filename, index=False, encoding='utf-8-sig')
    
    logger.info(f"✅ 수집 완료: 총 {len(final_df)}건")
    logger.info(f"파일 저장: {filename}")
    
    # 데이터 미리보기
    print("\n=== 수집된 데이터 미리보기 ===")
    print(f"총 데이터 수: {len(final_df)}")
    print(f"컬럼: {list(final_df.columns)}")
    if len(final_df) > 0:
        print("\n상위 5개 데이터:")
        print(final_df[['title', 'type', 'sido', 'location']].head())
    
    return final_df

# 실행
if __name__ == "__main__":
    df = main()

INFO:__main__:국가유산 데이터 수집 시작
전국 시도코드 순회:   0%|           | 0/17 [00:00<?, ?it/s]INFO:__main__:서울(11) 지역 수집 시작
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=1
INFO:__main__:API 응답 테스트:
INFO:__main__:Status Code: 200
INFO:__main__:Content-Type: application/xml; charset=utf-8
INFO:__main__:Response length: 28058
INFO:__main__:First 200 chars: <?xml version="1.0" encoding="utf-8"?>







<result>
<totalCnt>2115</totalCnt>
<pageUnit>50</pageUnit>
<pageIndex>1</pageIndex>	

<item>
<sn>1</sn>
<no>1</no>
<ccmaName><![CDATA[국보]]
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 1페이지: 50건 처리 중
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=2
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 2페이지: 50건 처리 중
INFO:__main__:총 100건 수집 완료
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=3
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 3페이지: 50건 처리 중
INFO:__main__:국가유산 목록 요청: 시도코드=11, 페이지=4
INFO:__main__:XML 응답을 파싱합니다.
INFO:__main__:서울 4페이지: 50건 처리 중
ERROR:__main__:XML 파싱 오류: mismatched tag: line 68, column 69
INFO:__main__:총 200건 수집 완


=== 수집된 데이터 미리보기 ===
총 데이터 수: 15515
컬럼: ['title', 'title_hanja', 'type', 'category', 'location', 'sido', 'description', 'designation_date', 'owner', 'manager', 'era', 'quantity', 'longitude', 'latitude', 'image_url', 'video_url', 'heritage_code', 'management_number', 'sido_code']

상위 5개 데이터:
               title type sido  \
0             서울 숭례문   국보   서울   
1       서울 원각사지 십층석탑   국보   서울   
2  서울 북한산 신라 진흥왕 순수비   국보   서울   
3        청자 사자형뚜껑 향로   국보   서울   
4         청자 어룡형 주전자   국보   서울   

                                            location  
0  \n\t\t\t\n\t             \n\t                 ...  
1  \n\t\t\t\n\t             \n\t                 ...  
2  \n\t\t\t\n\t             \n\t                 ...  
3  \n\t\t\t\n\t             \n\t                 ...  
4  \n\t\t\t\n\t             \n\t                 ...  
